# 🔋 キープアライブ ON/OFF + 外部受信検証（GPU 不要）

GPU を使わず、以下を確認するノート。

1. keepalive の **残り時間表示・リセット・停止**
2. auth_proxy.py + cloudflared quick tunnel での **外部からのアクセス受信**



---
### 🧩 共通ランタイム（tunnel / keepalive / proxy）

`runtime/` の真実源をビルダーが連結して埋め込んでいます。



In [ ]:
# ============================================================
# 共通ランタイム（runtime/*.py から自動生成。手で編集しない）
# proxy / tunnel / keepalive / verify の配管をここに集約する。
# ============================================================
# ── runtime/net.py ────────────────────────────────────────
"""トンネル URL 取得と HTTP 疎通の共通処理（生成コードへ埋め込む）。"""
import json
import re
import time
import urllib.error
import urllib.request

# hosts は表駆動。URL の出所（ログ解析 / 管理API）と待ち時間だけが違う。
TUNNEL_HOSTS = {
    "cloudflared": {
        "kind": "log",
        "url_re": re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com"),
        "wait_seconds": 90,
    },
    "ngrok": {
        "kind": "api",
        "api_url": "http://127.0.0.1:4040/api/tunnels",
        "wait_seconds": 60,
    },
}


def kc_probe_url(host, log_path=None):
    """現在の公開URLを返す（無ければ ""）。host は cloudflared / ngrok。"""
    spec = TUNNEL_HOSTS[host]
    if spec["kind"] == "api":
        try:
            with urllib.request.urlopen(spec["api_url"], timeout=3) as r:
                d = json.loads(r.read().decode())
            for t in d.get("tunnels", []):
                u = t.get("public_url", "")
                if u.startswith("https://"):
                    return u
        except Exception:
            return ""
        return ""
    if not log_path:
        return ""
    try:
        with open(log_path) as f:
            m = spec["url_re"].search(f.read())
        return m.group(0) if m else ""
    except OSError:
        return ""


def kc_wait_url(host, log_path=None, seconds=None, interval=1, progress_every=10):
    """公開URLが出るまで待つ。seconds 未指定なら host ごとの既定値。"""
    if seconds is None:
        seconds = TUNNEL_HOSTS[host]["wait_seconds"]
    for i in range(seconds):
        url = kc_probe_url(host, log_path)
        if url:
            return url
        if progress_every and i % progress_every == progress_every - 1:
            print(f"⏳ URL 待ち ({i + 1}/{seconds})...")
        time.sleep(interval)
    return ""


def kc_wait_http(url, attempts=20, interval=3, timeout=10, opener=None):
    """200 が返るまで待つ。成功したら経過秒（(i+1)*interval）、失敗なら None。"""
    opener = opener or urllib.request.urlopen
    for i in range(attempts):
        try:
            with opener(url, timeout=timeout) as r:
                if r.status == 200:
                    return (i + 1) * interval
        except Exception:
            pass
        time.sleep(interval)
    return None


def kc_retry(fn, tries=6, delay=5, skip_http_errors=True):
    """一過性の通信エラーを吸収する。HTTP 応答は「答え」なので既定で再試行しない。"""
    last = None
    for i in range(max(1, tries)):
        try:
            return fn()
        except urllib.error.HTTPError as e:
            if skip_http_errors:
                raise
            last = e
        except Exception as e:
            last = e
        if i + 1 < tries:
            wait = delay * (i + 1)
            print(f"  ⚠ 一時的な通信エラー（{i + 1}/{tries}）: {last} "
                  f"→ {wait}s 待って再試行")
            time.sleep(wait)
    raise last


def kc_post_json(url, payload, timeout=60, headers=None, opener=None):
    """JSON POST のレスポンスを返す（ストリームは呼び出し側で読む）。"""
    h = {"Content-Type": "application/json",
         "ngrok-skip-browser-warning": "true"}
    if headers:
        h.update(headers)
    req = urllib.request.Request(
        url, data=json.dumps(payload).encode(), headers=h, method="POST")
    return (opener or urllib.request.urlopen)(req, timeout=timeout)

# ── runtime/keepalive.py ──────────────────────────────────
"""キープアライブ設定ファイルと /stop の共通処理（生成コードへ埋め込む）。"""
import json
import os
import urllib.request


def kc_read_conf(path, default_minutes=60, max_minutes=480,
                 on_invalid="fail", fail=None, quiet_clamp=False):
    """(run, minutes, mtime) を返す。RUN=0 は (0, default_minutes, mtime)。

    on_invalid: "fail" は fail(msg)（未指定なら ValueError）、
                "zero" は停止扱い（RUN=0 と同じ）。
    quiet_clamp: proxy が書く既定値を上限で頭打ちにするとき黙る。
    """
    try:
        with open(path) as f:
            d = dict(l.strip().split("=", 1) for l in f
                     if l.strip() and "=" in l)
        run = int(d.get("RUN", 1))
        minutes = int(d.get("MINUTES", default_minutes))
    except OSError:
        run, minutes = 1, default_minutes
    except ValueError:
        run, minutes = 1, default_minutes
    try:
        mtime = os.path.getmtime(path)
    except OSError:
        mtime = None
    if run <= 0:
        return 0, default_minutes, mtime
    if minutes <= 0:
        msg = f"KEEPALIVE_MINUTES={minutes} は無効（1〜{max_minutes} で指定）"
        if on_invalid == "zero":
            return 0, default_minutes, mtime
        if fail:
            fail(msg)
        raise ValueError(msg)
    if minutes > max_minutes:
        if not (quiet_clamp and minutes == default_minutes):
            print(f"⚠ KEEPALIVE_MINUTES={minutes} → {max_minutes} に制限")
        minutes = max_minutes
    return run, minutes, mtime


def kc_write_conf(path, run, minutes=None):
    with open(path, "w") as f:
        f.write(f"RUN={run}\n")
        if minutes is not None:
            f.write(f"MINUTES={minutes}\n")


def kc_conf_mtime(path):
    """keepalive.conf の更新時刻。proxy がチャット送信ごとに書き直す。"""
    try:
        return os.path.getmtime(path)
    except OSError:
        return 0.0


def kc_stop_via_proxy(port, auth=None, timeout=30):
    headers = {"X-Kc-Stop": "1"}
    if auth:
        headers["Authorization"] = auth
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/stop", data=b"", method="POST",
        headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, json.loads(r.read())

# ── runtime/proxy.py ──────────────────────────────────────
"""auth_proxy の起動と認証なし検査（生成コードへ埋め込む）。"""
import subprocess
import time
import urllib.error
import urllib.request


def kc_start_proxy(port, upstream, chat_html, proxy_path,
                   extra_args=(), env=None, log_path="/tmp/auth_proxy.log",
                   wait_seconds=30, label="プロキシ", fail=None):
    """auth_proxy.py を起動し /healthz を待つ。失敗時は fail() か RuntimeError。"""
    subprocess.run(["pkill", "-f", "auth_proxy.py"], capture_output=True)
    time.sleep(1)
    cmd = ["python3", proxy_path, "--port", str(port),
           "--upstream", upstream, "--chat-html", chat_html, *extra_args]
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, env=env, stdout=log, stderr=log)
    print(f"🔄 proxy PID: {proc.pid}")

    elapsed = None
    for i in range(wait_seconds):
        try:
            with urllib.request.urlopen(
                    f"http://127.0.0.1:{port}/healthz", timeout=3) as r:
                if r.status == 200:
                    elapsed = i + 1
                    break
        except Exception:
            pass
        time.sleep(1)
    if elapsed is None:
        try:
            print(open(log_path).read()[-2000:])
        except OSError:
            pass
        msg = f"{label}の起動に失敗しました（{log_path}）"
        if fail:
            fail(msg, log_path)
        raise RuntimeError(msg)
    print(f"✅ {label}起動完了（{elapsed}s）")
    return proc


def kc_assert_no_auth(port, path="/api/tags", fail=None, timeout=10):
    """401/403 なら認証が有効 = 設定ミス。上流未起動の 5xx は正常。"""
    req = urllib.request.Request(f"http://127.0.0.1:{port}{path}")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            print(f"  ✅ 認証なし → HTTP {r.status}")
            return True
    except urllib.error.HTTPError as e:
        if e.code in (401, 403):
            msg = f"認証なしで HTTP {e.code} — 認証が有効になっています"
            if fail:
                fail(msg)
            raise RuntimeError(msg)
        print(f"  ✅ 認証なし → HTTP {e.code}（401 でない）")
        return True
    except Exception as e:
        msg = f"プロキシ検査に失敗: {type(e).__name__}: {e}"
        if fail:
            fail(msg)
        raise



---

## セル 1: keepalive 自動検証ループ



In [ ]:
# 🔋 キープアライブ ON/OFF 自動検証（GPU 不要）
# /tmp/keepalive.conf を自動操作して、残り時間表示 + リセット + 停止を検証。
import time

MAX_MINUTES = 480
KEEPALIVE_CONF = "/tmp/keepalive.conf"
KEEPALIVE_KEEPALIVE = 60
SLEEP_SEC = 3                     # GPU なしなので 3 秒ごと（実機では 60）
RESET_AT = 10                     # 10 秒後にチャット送信シミュレーション
STOP_AT = 25                      # 25 秒後に停止ボタンシミュレーション


def _read_keepalive_conf():
    """RUN=1/0 と MINUTES を読み取り（デフォルトは 1 と KEEPALIVE_KEEPALIVE）。"""
    run, minutes, _mtime = kc_read_conf(
        KEEPALIVE_CONF, KEEPALIVE_KEEPALIVE, MAX_MINUTES, on_invalid="zero")
    return run, minutes


def _reset_keepalive():
    """チャット送信ごとに時間リセット（ON 維持）。"""
    try:
        kc_write_conf(KEEPALIVE_CONF, 1, KEEPALIVE_KEEPALIVE)
        print("🔄 keepalive リセット → 60 分")
        return True
    except Exception as e:
        print(f"⚠ リセット失敗: {e}")
        return False


def _stop_keepalive():
    """停止ボタン：RUN=0 にする。"""
    try:
        kc_write_conf(KEEPALIVE_CONF, 0)
        print("🛑 keepalive 停止 → RUN=0")
        return True
    except Exception as e:
        print(f"⚠ 停止失敗: {e}")
        return False


# 初期化
_reset_keepalive()
start = time.time()
last_reset = start
RUN, KEEPALIVE_MINUTES = _read_keepalive_conf()
print(f"初期: RUN={RUN} MINUTES={KEEPALIVE_MINUTES}")
print(f"🔋 ON。{KEEPALIVE_KEEPALIVE}分間維持")
print(f"⏱ {RESET_AT}秒後にリセット、{STOP_AT}秒後に停止")
print()

log = []
reset_done = stop_done = False
i = 0
while True:
    i += 1
    now = time.time()
    elapsed = now - start
    RUN, KEEPALIVE_MINUTES = _read_keepalive_conf()

    if not RUN:
        print(f"RUN=0 → {i}周目（{elapsed:.0f}秒経過）に終了")
        log.append(f"[{elapsed:.0f}s] RUN=0 → 終了")
        break

    # 自動イベント
    if not reset_done and elapsed >= RESET_AT:
        if _reset_keepalive():
            last_reset = time.time()
            reset_done = True
    if not stop_done and elapsed >= STOP_AT:
        if _stop_keepalive():
            stop_done = True

    # 残り時間（最後のリセットから）
    remaining_sec = max(0, KEEPALIVE_MINUTES * 60 - (now - last_reset))
    remaining_min = int(remaining_sec // 60)
    remaining_sec_only = int(remaining_sec % 60)

    line = f"[{elapsed:5.1f}s] alive | 残り {remaining_min:3d}分{remaining_sec_only:02d}秒"
    print(line)
    log.append(line)
    time.sleep(SLEEP_SEC)

report = "\n".join([
    f"result={'stopped' if not RUN else 'alive'}",
    f"RUN={RUN}",
    f"minutes={KEEPALIVE_MINUTES}",
    f"elapsed={elapsed:.0f}s",
    f"reset_done={reset_done}",
    f"stop_done={stop_done}",
    "",
] + log)

with open("keepalive_test_result.txt", "w") as f:
    f.write(report)
print()
print("== keepalive_test_result.txt 保存 ==")
print(report)


---

## セル 2: 外部受信テスト

auth_proxy.py + cloudflared quick tunnel を立てて、公開URLに自分自身からアクセス。

**自動で確認すること**

- `/healthz` → 200
- `/quota` → 200（`--no-quota` なので available=false）
- `POST /stop` → 200
- 停止後の `/healthz` → 接続失敗（プロキシが落ちた）



In [ ]:
# 🌐 外部受信テスト（GPU 不要・Internet On）
# auth_proxy.py + cloudflared quick tunnel で、インターネットからのアクセスを確認。
import base64, os, subprocess, time, urllib.request

PORT = 8080

PROXY_SRC_B64 = (
    "..."
)
CHAT_SRC_B64 = (
    "..."
)

# --- cloudflared 準備 ---
CF_BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(CF_BIN):
    print("cloudflared をダウンロード中...")
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, CF_BIN)
    os.chmod(CF_BIN, 0o755)
    print(f"✅ {CF_BIN} ダウンロード完了")

# --- auth_proxy.py と chat.html をデコード ---
with open("/tmp/auth_proxy.py", "wb") as f:
    f.write(base64.b64decode(PROXY_SRC_B64))
with open("/tmp/llama-chat.html", "wb") as f:
    f.write(base64.b64decode(CHAT_SRC_B64))
print("✅ proxy / chat ui デコード完了")

# --- プロキシ起動（/healthz まで待つ） ---
proxy_proc = kc_start_proxy(
    PORT, "http://127.0.0.1:11434", "/tmp/llama-chat.html", "/tmp/auth_proxy.py",
    extra_args=("--no-quota",))
print(f"🚀 proxy PID: {proxy_proc.pid}")

# --- cloudflared 起動 ---
TUNNEL_LOG = "/tmp/cloudflared.log"
subprocess.run(["pkill", "-x", "cloudflared"], capture_output=True)
time.sleep(1)
with open(TUNNEL_LOG, "w") as log:
    cf_proc = subprocess.Popen([CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                                stdout=log, stderr=log)
print(f"🚀 cloudflared PID: {cf_proc.pid}")

# --- 公開URL取得 ---
public_url = kc_wait_url("cloudflared", TUNNEL_LOG)
if not public_url:
    print("cloudflared ログ（末尾 2000 文字）:")
    try:
        print(open(TUNNEL_LOG).read()[-2000:])
    except FileNotFoundError:
        print("ログファイルなし")
    raise RuntimeError("公開URL取得失敗")

print(f"🌍 Public URL: {public_url}")

# --- DNS 伝播待ち（healthz が通るまで） ---
print()
print("DNS 伝播待ち...")
_elapsed = kc_wait_http(f"{public_url}/healthz", attempts=10, interval=3)
print(f"  {'✅ 疎通 OK' if _elapsed is not None else '⚠ 未確立'}")

# --- 接続情報保存 ---
with open("public_url.txt", "w") as f:
    f.write(public_url + "\n")
with open("connection_info.txt", "w") as f:
    f.write(f"URL: {public_url}\n")
    f.write(f"proxy PID: {proxy_proc.pid}\n")
    f.write(f"cloudflared PID: {cf_proc.pid}\n")

# --- 外部疎通テスト（自分自身へアクセス） ---
print()
print("=" * 60)
print(f"  公開URL: {public_url}")
print("=" * 60)
print()

print("外部受信テスト：公開URLに自分自身からアクセス...")
for path in ["/healthz", "/quota"]:
    ok = False
    for attempt in range(1, 11):
        try:
            with urllib.request.urlopen(f"{public_url}{path}", timeout=15) as r:
                body = r.read().decode()[:200]
                print(f"GET {path}: {r.status} {body}")
                ok = True
                break
        except Exception as e:
            print(f"GET {path} attempt {attempt}/10: {type(e).__name__}")
            time.sleep(3)
    if not ok:
        print(f"GET {path}: 10 回リトライ後も失敗")

# --- 停止テスト ---
# /stop は POST + X-Kc-Stop: 1 が必須（無いと 405 でプロキシは落ちない）。
print()
print("POST /stop ...")
try:
    req = urllib.request.Request(f"{public_url}/stop", method="POST", data=b"",
                                 headers={"X-Kc-Stop": "1"})
    with urllib.request.urlopen(req, timeout=10) as r:
        body = r.read().decode()
        print(f"/stop: {r.status} {body[:200]}")
except Exception as e:
    print(f"/stop error: {type(e).__name__}: {e}")

# /stop 後に再アクセス → 落ちているはず
time.sleep(3)
print()
print("停止後疎通テスト...")
try:
    with urllib.request.urlopen(f"{public_url}/healthz", timeout=5) as r:
        print(f"停止後 /healthz: {r.status} （まだ生きている）")
except Exception as e:
    print(f"停止後 /healthz error: {type(e).__name__} ✅ 落ちた")

# 掃除
proxy_proc.terminate()
cf_proc.terminate()
print("プロセス終了")


---

## セル 3: 結果確認



In [ ]:
# 結果ファイルを表示
for fname in ["keepalive_test_result.txt", "public_url.txt", "connection_info.txt"]:
    print(f"=== {fname} ===")
    try:
        print(open(fname).read())
    except FileNotFoundError:
        print("まだありません")
    print()
